# Cell communication analysis: mcTECs

In [ ]:
import os
import sys
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import hdf5plugin
#import liana as li

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

# Import custom scripts
from utils import get_latest_version,update_obs,freq_by_donor
from anno_levels import get_ct_levels, get_ct_palette, age_group_levels, age_group_palette
from plotting.utils import plot_grouped_boxplot, calc_figsize

In [ ]:
# Define columns
col_cell_type_broad = 'taa_l3'
col_cell_type_fine = 'taa_l4'
# col_cell_type_broad_levels = get_ct_levels(col_cell_type_broad, taa_l1 = ['T', 'NK'])
# col_cell_type_fine_levels = get_ct_levels(col_cell_type_fine, taa_l1 = ['T', 'NK'])
col_age_group = 'age_group'
col_age_group_levels = eval(f'{col_age_group}_levels')

In [ ]:
ct_anno['taa_l3'].unique()

In [ ]:
ct_anno = pd.read_csv(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv', index_col = 0)

mctec_barcodes = ct_anno[ct_anno['taa_l3'].isin(['mcTEC', 'cTEC'])].index.tolist()

# Load adata
object_version = 'v5_2025-04-03'
adata = ad.read_h5ad(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_{object_version}.zarr', backed = 'r')

# Subset adata to only include mcTEC cells
adata_mctec = adata[mctec_barcodes, :].to_memory()

# Add annotations to adata
adata_mctec.obs = adata_mctec.obs.join(ct_anno, how = 'left')
adata_mctec = adata_mctec[(adata_mctec.obs['anno_status'] == 'include') & (adata_mctec.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_mctec, latest_meta, on = 'index', ignore_warning = True)

adata_mctec.shape

Filter genes to receptors/ligands expressed in mcTEC:

In [ ]:
# Load CellPhoneDB database (v5.0.0)
ia_db = pd.read_csv(f'{data_path}/references/ccc_databases/cellphoneDB_v5.0.0_interaction_input.csv')
ia_db[['ligand', 'receptor']] = ia_db['interactors'].str.rsplit('-', n=1, expand=True)
ia_db['receptor'] = ia_db['receptor'].str.split('+')
ia_db = ia_db.explode('receptor')
ia_db.head()

In [ ]:
adata_mctec = adata_mctec[:, adata_mctec.var_names.isin(ia_db['ligand'].unique().tolist() + ia_db['receptor'].unique().tolist())]
adata_mctec.var['frac_expressed'] = np.array((adata_mctec.X > 0).sum(axis=0) / adata_mctec.n_obs).flatten()

adata_mctec

In [ ]:
plt.figure(figsize=calc_figsize(width=80, height=60))
sns.histplot(adata_mctec.var['frac_expressed'], bins=50, kde=False, color='steelblue')
plt.xlabel('Fraction expressed')
plt.ylabel('Number of genes')
plt.title('Histogram of frac_expressed in adata_mctec')
plt.tight_layout()
plt.show()

In [ ]:
lr_oi = adata_mctec.var_names[adata_mctec.var['frac_expressed'] > 0.1].tolist()
print(f'Number of ligands/receptors expressed in >10% of cells: {len(lr_oi)}')

ia_oi = ia_db[(ia_db['receptor'].isin(lr_oi)) & (ia_db['ligand'].isin(adata.var_names))]
print(f'Number of interactions with receptor expressed in >10% of cells: {ia_oi.shape[0]}')

all_lr = set(ia_oi['ligand'].unique().tolist() + ia_oi['receptor'].unique().tolist())
print(f'Total number of unique ligands/receptors in CellPhoneDB: {len(all_lr)}')

Identify cell types which could communicate with mcTEc based on niche assignments:

In [ ]:
niche_comp = pd.read_csv('/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/mcTEC_niche_k6/mctec_niche_k6_taa_l3_comb_composition_pct.csv', index_col=0)
niche_recovery = pd.read_csv('/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/mcTEC_niche_k6/mctec_niche_k6_per_donor_pct.csv', index_col=0)
niche_recovery['age_group'] = ['adult' if x.startswith('A') else 'paed' if x.startswith('Z') else 'infant' for x in niche_recovery.index]
niche_recovery

In [ ]:
niche_recovery.groupby('age_group').mean().T.plot(kind='bar', figsize=calc_figsize(width=80, height=60))

In [ ]:
niche_groups = niche_recovery.groupby('age_group').mean().T

pct_thres = 10
shared_niches = niche_groups.loc[(niche_groups['adult'] > pct_thres) & (niche_groups['paed'] > pct_thres) & (niche_groups['infant'] > pct_thres)].index.tolist()
adult_niches = [x for x in niche_groups.index if x not in shared_niches]

print(f'Number of shared niches: {len(shared_niches)}')
print(f'Number of adult niches: {len(adult_niches)}')

Shared niches -> take from paed and infant

In [ ]:
# Proportion of niches in each age group
niche_ratios_shared = niche_groups.loc[shared_niches, :]
niche_ratios_shared = niche_ratios_shared.div(niche_ratios_shared.sum(axis=0), axis=1)
niche_ratios_shared

In [ ]:
niche_comp_shared = niche_comp.loc[shared_niches, :] / 100
niche_comp_shared = pd.DataFrame(niche_ratios_shared.values.T @ niche_comp_shared.values, columns=niche_comp_shared.columns, index=niche_ratios_shared.columns)
niche_comp_shared

Adult niches -> take from adult

In [ ]:
# Proportion of niches in each age group
niche_ratios_adult = niche_groups.loc[adult_niches, :]
niche_ratios_adult = niche_ratios_adult.div(niche_ratios_adult.sum(axis=0), axis=1)
niche_ratios_adult

In [ ]:
niche_comp_adult = niche_comp.loc[adult_niches, :] / 100
niche_comp_adult = pd.DataFrame(niche_ratios_adult.values.T @ niche_comp_adult.values, columns=niche_comp_adult.columns, index=niche_ratios_adult.columns)
niche_comp_adult

In [ ]:
# Use adult age group for adult niche
niche_comp_combined = niche_comp_shared.copy()
niche_comp_combined.loc['adult'] = niche_comp_adult.loc['adult']

niche_comp_combined

In [ ]:
ct_prop_threshold = 0.02

ct_per_niche = {
    a: niche_comp_combined.columns[niche_comp_combined.loc[a] >= ct_prop_threshold].tolist()
    for a in niche_comp_combined.index
}
all_ct_oi = set([ct for ct_list in ct_per_niche.values() for ct in ct_list])

import pprint
pprint.pprint(ct_per_niche, compact=True, width=120)
pprint.pprint(all_ct_oi, compact=True, width=120)


In [ ]:
[c for c in all_ct_oi if c not in ct_anno['taa_l3'].values]

In [ ]:
# Subset adata
adata_sub_barcodes = ct_anno[ct_anno['taa_l3'].isin(all_ct_oi)].index.tolist()
adata_sub = adata[adata_sub_barcodes,adata.var_names.isin(all_lr)].to_memory()

# Add annotations to adata
adata_sub.obs = adata_sub.obs.join(ct_anno, how = 'left')
adata_sub = adata_sub[(adata_sub.obs['anno_status'] == 'include') & (adata_sub.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_sub, latest_meta, on = 'index', ignore_warning = True)

adata_sub.obs['cell_type'] = adata_sub.obs['taa_l3']

adata_sub.obs['cell_type'].value_counts()

Inlcude donors with 50 mcTECs

In [ ]:
# Filter out donors with fewer than 50 mcTECs
min_cells = 50
col_sample = 'donor'
ncells_by_sample = adata_sub[adata_sub.obs[col_cell_type_broad] == 'mcTEC'].obs[col_sample].value_counts().to_frame(name = 'n_cells')
print('Removing {} donors with fewer than {} cells'.format(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].shape[0], min_cells))
adata_sub = adata_sub[~adata_sub.obs[col_sample].isin(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].index)]

In [ ]:
# Level to cell type assignment
ct_levels = [c for c in get_ct_levels('taa_l3') if c in all_ct_oi]
import pprint
pprint.pprint(ct_levels, compact = True)

In [ ]:
# Niche composition for each cell type by age group
ct_freq = niche_comp_combined.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index()
ct_freq = ct_freq.loc[ct_freq['cell_type'].isin(ct_levels), :]
ct_freq

In [ ]:
interactions_oi = ia_oi[['ligand', 'receptor']].drop_duplicates().apply(lambda x: (x['ligand'], x['receptor']), axis = 1).tolist()
interactions_oi 

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
adata_sub.obs['ct_donor'] = adata_sub.obs['donor'] + '__' + adata_sub.obs['cell_type']
ct_by_donor_dummy = pd.get_dummies(adata_sub.obs['ct_donor'])
n_cells_per_ct_donor = ct_by_donor_dummy.sum(axis=0)
X_bool = (adata_sub.X > 0).astype(int)

frac_by_ct = ct_by_donor_dummy.values.T * X_bool
frac_by_ct = frac_by_ct / n_cells_per_ct_donor.to_numpy()[:, None]

frac_by_ct

In [ ]:
frac_by_ct_df = pd.DataFrame(frac_by_ct, index=ct_by_donor_dummy.columns, columns=adata_sub.var_names)
frac_by_ct_df['cell_type'] = frac_by_ct_df.index.str.split('__').str[-1]
frac_by_ct_df['donor'] = frac_by_ct_df.index.str.split('__').str[0]
frac_by_ct_df = frac_by_ct_df.melt(id_vars=['cell_type', 'donor'], var_name='gene', value_name='frac_expressed')
frac_by_ct_df.head()

In [ ]:
lr_frac_expressed = ia_oi[['ligand', 'receptor']].drop_duplicates() \
    .merge(frac_by_ct_df, left_on='ligand', right_on='gene', how='left').rename(columns={'frac_expressed': 'ligand_props', 'cell_type' : 'source'}).drop(columns = 'gene') \
    .merge(frac_by_ct_df, left_on=['receptor', 'donor'], right_on=['gene', 'donor'], how='left').rename(columns={'frac_expressed': 'receptor_props', 'cell_type': 'target'}).drop(columns = 'gene') 
lr_frac_expressed['age_group'] = lr_frac_expressed['donor'].map(adata_sub.obs[['donor', 'age_group']].drop_duplicates().set_index('donor')['age_group'])
lr_frac_expressed['interaction'] = lr_frac_expressed['ligand'] + '->' + lr_frac_expressed['receptor']
lr_frac_expressed['cells'] = lr_frac_expressed['source'] + '->' + lr_frac_expressed['target']
lr_frac_expressed = lr_frac_expressed.merge(ct_freq[['age_group', 'cell_type', 'niche_prop']], left_on=['age_group', 'source'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'source_prop'}).drop(columns=['cell_type']) \
    .merge(ct_freq[['age_group', 'cell_type','niche_prop']], left_on=['age_group', 'target'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'target_prop'}).drop(columns=['cell_type'])
#lr_frac_expressed['age_group'] = np.where(lr_frac_expressed['age_group'] == 'adult', 'adult', 'paed') # Combine paed and infant into one group for comparison with of paed shared niche and adult-specific niche

lr_frac_expressed

In [ ]:
# Subset to interactions with mcTEC as target
lr_frac_expressed = lr_frac_expressed[lr_frac_expressed['target'].isin(['mcTEC','cTEC'])].dropna()
lr_frac_expressed.shape

In [ ]:
lr_frac_expressed.to_csv(f'{data_path}/analyses/cellComm/mcTEC/combined_niche_lr_expressed_by_interaction_and_celltype.csv', index=False)

Replace source/target proportion with niche-specific proportions

In [ ]:
# Calculate mean properties (using all samples, not just TOT sort)
lr_frac_expressed['tot_prob'] = lr_frac_expressed['ligand_props'] * lr_frac_expressed['receptor_props'] * lr_frac_expressed['source_prop'] * lr_frac_expressed['target_prop']
mean_props = lr_frac_expressed.groupby(['interaction', 'cells', 'age_group']).agg({'ligand_props' : 'mean', 'receptor_props' : 'mean', 'tot_prob': 'mean'}).reset_index()
mean_props = mean_props.pivot(index = ['interaction', 'cells'], columns = 'age_group', values = ['ligand_props', 'receptor_props','tot_prob']).reset_index()
mean_props.columns = ['_'.join(col).strip() for col in mean_props.columns.values]
mean_props = mean_props.rename(columns = {'interaction_': 'interaction', 'cells_': 'cells'})
mean_props.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

# Filter interactions
# Select interactions with at least one age group with expression > 0.1
target_age_group = ['paed', 'infant']
min_expr = 0.1

# Keeping any interaction with at least one age group with expression > in expr
#ia_to_keep = mean_props.loc[mean_props[['ligand_props_' + ag for ag in target_age_group] + ['receptor_props_' + ag for ag in target_age_group]].ge(min_expr).all(axis=1)][['interaction', 'cells']]

# Keeping interactions where L and R are both expressed at min expr in the same age group
ia_to_keep = []
for g in target_age_group:
    ia_to_keep.append(mean_props.loc[mean_props[[f'ligand_props_{g}', f'receptor_props_{g}']].ge(min_expr).all(axis=1)][['interaction', 'cells']])
ia_to_keep = pd.concat(ia_to_keep, axis = 0).drop_duplicates()

print(f'Number of interactions to keep: {ia_to_keep.shape[0]}')

donor_n = adata.obs[['donor', col_age_group]].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']
# Define a function to process a single cell
def process_cells(ct):
    ias = ia_to_keep[ia_to_keep['cells'] == ct]['interaction'].unique()
    test_ranksum = []
    for ia in ias:
        test_data = lr_frac_expressed[(lr_frac_expressed['interaction'] == ia) & (lr_frac_expressed['cells'] == ct)][['donor', col_age_group, 'tot_prob']].dropna(subset=['tot_prob']).copy()
        group1 = test_data[test_data['age_group'] == target_age_group[0]]['tot_prob'].to_numpy()
        if group1.shape[0] < donor_n[target_age_group[0]]:
            group1 = np.concatenate((group1, np.repeat(0, repeats=donor_n[target_age_group[0]] - group1.shape[0])))
        group2 = test_data[test_data['age_group'] == target_age_group[1]]['tot_prob'].to_numpy()
        if group2.shape[0] < donor_n[target_age_group[1]]:
            group2 = np.concatenate((group2, np.repeat(0, repeats=donor_n[target_age_group[1]] - group2.shape[0])))
        test_res = ranksums(group1, group2)
        test_ranksum.append([ia, ct, test_res[0], test_res[1]])
    test_ranksum = pd.DataFrame(test_ranksum, columns=['interaction', 'cells', 'statistic', 'pvalue'])
    test_ranksum['padj'] = multipletests(test_ranksum['pvalue'], method='fdr_bh')[1]
    return test_ranksum

# Use ThreadPoolExecutor with 4 threads to process cells in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_cells, ia_to_keep['cells'].unique()))

# Combine results
ranksums_res = pd.concat(results, axis=0)
ranksums_res.to_csv(f'{data_path}/analyses/cellComm/mcTEC/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv')

ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res = pd.read_csv(f'{data_path}/analyses/cellComm/mcTEC/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0)
ranksums_res[['source', 'target']] = ranksums_res['cells'].str.split('->', expand=True)
ranksums_res[['ligand', 'receptor']] = ranksums_res['interaction'].str.split('->', expand=True)
ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered = ranksums_res[ranksums_res['padj'] < 0.05]
ranksums_res_filtered['ia_change_direction'] = np.where(ranksums_res_filtered['statistic'] > 0, 'up', 'down')
ranksums_res_filtered.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered['source'].value_counts()

In [ ]:
# Adult niche vs infant niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult niche vs paed niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Paed niche vs infant niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult niche vs paed niche
pprint.pprint(ranksums_res_filtered.groupby(['cells','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

## Plotting

In [ ]:
contrasts = [ ('adult', 'paed'), ('adult', 'infant')]

combined_niche_ranksums = [pd.read_csv(f'{data_path}/analyses/cellComm/mcTEC/combined_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0) for target_age_group in contrasts]
nrows = [df.shape[0] for df in combined_niche_ranksums]

combined_niche_ranksums = pd.concat(combined_niche_ranksums, axis = 0)
combined_niche_ranksums['contrasts'] = np.repeat([f'{c[0]}_vs_{c[1]}' for c in contrasts], nrows)

combined_niche_ranksums

In [ ]:
contrasts = [('adult', 'paed'), ('adult', 'infant'), ('paed', 'infant')]

shared_niche_ranksums = [pd.read_csv(f'{data_path}/analyses/cellComm/mcTEC/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0) for target_age_group in contrasts]
nrows = [df.shape[0] for df in shared_niche_ranksums]

shared_niche_ranksums = pd.concat(shared_niche_ranksums, axis = 0)
shared_niche_ranksums['contrasts'] = np.repeat([f'{c[0]}_vs_{c[1]}' for c in contrasts], nrows)

shared_niche_ranksums

In [ ]:
# Lost from thymocytes to mcTEC
MODULES = {
    
    'thymocyte_lost' : {
    "TNF_superfamily_lymphotoxin": [
        "LTB->LTBR",
        "TNFSF10->TNFRSF10B",
        "TNFSF10->TNFRSF10D",
        "APP->TNFRSF21",
    ],
    "TGFB": [
        "TGFB1->TGFBR2",
        "TGFB1->TGFBR3",
        "TGFB1->ITGAV",
    ],
    "steroid_conversion": [
        "HSD17B12->AR",
        "SRD5A3->AR",
    ],
    "sterol_lipid_nuclear_receptor": [
        "DHCR7->NR1H2",
        "DHCR7->RORA",
        "LIPA->RORA",
        "PTGES2->PTGER4",
        "LTA4H->LTB4R2",
    ],
    "guidance_semaphorin_ephrin": [
        "SEMA4D->PLXNB1",
        "SEMA4D->PLXNB2",
        "SEMA7A->ITGB1",
        "EFNA1->EPHA1",
        "EFNA1->EPHA4",
    ],
    "adhesion_junctional": [
        "JAM3->JAM3",
        "MPZL1->MPZL1",
        "DSG2->DSC3",
        "CD96->NECTIN1",
    ],
    "matrix": [
        "COL24A1->ITGA2",
        "COL24A1->ITGB1",
    ],
    "growth_factor_other": [
        "AREG->EGFR",
        "CD248->IL6R",
        "VEGFB->NRP1",
        "PRNP->ADGRG6",
    ],

    },
# ---------------------------------------------------------------------------
# 2. Gained from Fb-perilo onto mcTEC
# ---------------------------------------------------------------------------

'stromal_gained' : {
    "network_bm_collagen_integrin": [
        "COL4A1->ITGA2",
        "COL4A1->ITGB1",
        "COL4A2->ITGA2",
        "COL4A2->ITGB1",
        "COL6A1->ITGA2",
        "COL6A1->ITGB1",
        "COL6A2->ITGA2",
        "COL6A2->ITGB1",
        "COL6A3->ITGB1",
        "COL18A1->ITGB1",
    ],
    "laminin_matricellular": [
        "LAMC1->ITGA6",
        "LAMC1->ITGB1",
        "THBS1->ITGA3",
        "THBS1->ITGAV",
        "THBS1->ITGB1",
        "PLAUR->ITGAV",
        "PLAUR->ITGB1",
        "PLA2G2A->ITGB1",
        "VCAM1->ITGB1",
    ],
    "notch": [
        "JAG1->NOTCH2",
        "DLL1->NOTCH2",
        "JAG1->CD46",
        "JAG1->VASN",
    ],
    "retinoid": [
        "ALDH1A1->RXRA",
    ],
    "growth_factors": [
        "FGF7->FGFR2",
        "FGF10->FGFR2",
        "HGF->MET",
        "IGF1->IGF1R",
        "IGF1->ITGA6",
    ],
    "TGFB": [
        "TGFB1->TGFBR2",
        "TGFB1->TGFBR3",
    ],
    "steroid_conversion": [
        "HSD17B12->AR",
    ],
    "wnt_noncanonical": [
        "WNT5A->FZD6",
        "WNT5A->LRP5",
        "WNT5A->ROR2",
        "WNT5A->RYK",
        "WNT11->FZD6",
        "WNT11->LRP5",
    ],
    "il6_axis": [
        "IL6->IL6R",
        "IL6->IL6ST",
        "CD248->IL6R",
    ],
    "TNF_superfamily": [
        "TNFSF10->TNFRSF10B",
        "TNFSF10->TNFRSF10D",
        "TNFSF12->TNFRSF12A",
    ],
    "vascular": [
        "VEGFA->NRP1",
        "VEGFB->NRP1",
        "PGF->NRP1",
    ],
    "lipid_eicosanoid": [
        "LIPA->RORA",
        "PTGES2->PTGER4",
        "LTA4H->LTB4R",
        "PTGR1->LTB4R",
    ],
    "junctional": [
        "GJA1->GJA1",
    ],
    # Present in the source CellChat output but omitted from the summary
    # table; restored here because only APP->CD74 is on the exclusion list.
    "amyloid_precursor": [
        "APP->SORL1",
        "APP->TNFRSF21",
    ],
}
}

all_pairs = list(set([pair for module in MODULES.values() for pairs in module.values() for pair in pairs]))

In [ ]:
from plotting.colors import thyAgeing_colors
from matplotlib.backends.backend_pdf import PdfPages

# Keep only adhesion-related interactions if available
for module_name in MODULES.keys():
    module_pairs = [pair for pairs in MODULES[module_name].values() for pair in pairs]
    df_all = combined_niche_ranksums.loc[combined_niche_ranksums['interaction'].isin(module_pairs)].copy()

    pdf_path = f'{plots_path}/cellComm/mcTEC/combined_niches_{module_name.replace(" ", "_")}_ranksums_heatmaps.pdf'

    contrast_order = [c for c in ['adult_vs_paed'] if c in df_all['contrasts'].unique()]
    contrast_order += [c for c in df_all['contrasts'].dropna().unique() if c not in contrast_order]

    with PdfPages(pdf_path) as pdf:
        for contrast in contrast_order:
            df_sub = df_all[df_all['contrasts'] == contrast].copy()

            df_sub = df_sub.drop_duplicates(subset=['interaction', 'cells'])
            heat = df_sub.pivot(index='interaction', columns='cells', values='statistic')
            heat_anno = df_sub.pivot(index='interaction', columns='cells', values='padj').applymap(lambda x: '*' if x < 0.05 else '')

            order = heat.abs().max(axis=1).sort_values(ascending=False).index
            heat = heat.reindex(order)
            heat_anno = heat_anno.reindex(order)

            absmax = 4.0
            vmin, vmax = -absmax, absmax

            fig, ax = plt.subplots(figsize=calc_figsize(width=heat.shape[1] * 2.5 + 20, height=heat.shape[0] * 2.5 + 20))
            sns.heatmap(
                heat,
                cmap=sns.blend_palette([thyAgeing_colors['teal'], 'white', thyAgeing_colors['orange']], as_cmap=True),
                center=0,
                vmin=vmin,
                vmax=vmax,
                annot=heat_anno,
                fmt='',
                annot_kws={'fontsize': 8},
                cbar_kws={'label': 'ranksums statistic'},
                linewidths=0.0,
                xticklabels=1,   # force all x tick labels
                yticklabels=1,   # force all y tick labels
                ax=ax
            )

            # Explicitly set all ticks to prevent auto-skipping
            ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
            ax.set_xticklabels(heat.columns, rotation=90)
            ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
            ax.set_yticklabels(heat.index, rotation=0)

            ax.set_xlabel('cells')
            ax.set_ylabel(f'{module_name.title()} interaction')
            ax.set_title(f'{contrast} — statistic heatmap')
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print(f'Saved: {pdf_path}')

In [ ]:
from plotting.colors import thyAgeing_colors
from matplotlib.backends.backend_pdf import PdfPages

# Keep only adhesion-related interactions if available
for module_name in MODULES.keys():
    module_pairs = [pair for pairs in MODULES[module_name].values() for pair in pairs]
    df_all = shared_niche_ranksums.loc[shared_niche_ranksums['interaction'].isin(module_pairs)].copy()

    pdf_path = f'{plots_path}/cellComm/mcTEC/shared_niches_{module_name.replace(" ", "_")}_ranksums_heatmaps.pdf'

    contrast_order = [c for c in ['adult_vs_infant', 'adult_vs_paed'] if c in df_all['contrasts'].unique()]
    contrast_order += [c for c in df_all['contrasts'].dropna().unique() if c not in contrast_order]

    with PdfPages(pdf_path) as pdf:
        for contrast in contrast_order:
            df_sub = df_all[df_all['contrasts'] == contrast].copy()

            df_sub = df_sub.drop_duplicates(subset=['interaction', 'cells'])
            heat = df_sub.pivot(index='interaction', columns='cells', values='statistic')
            heat_anno = df_sub.pivot(index='interaction', columns='cells', values='padj').applymap(lambda x: '*' if x < 0.05 else '')

            order = heat.abs().max(axis=1).sort_values(ascending=False).index
            heat = heat.reindex(order)
            heat_anno = heat_anno.reindex(order)

            absmax = 4.0
            vmin, vmax = -absmax, absmax

            fig, ax = plt.subplots(figsize=calc_figsize(width=heat.shape[1] * 2.5 + 40, height=heat.shape[0] * 2.5 + 20))
            sns.heatmap(
                heat,
                cmap=sns.blend_palette([thyAgeing_colors['teal'], 'white', thyAgeing_colors['orange']], as_cmap=True),
                center=0,
                vmin=vmin,
                vmax=vmax,
                annot=heat_anno,
                fmt='',
                annot_kws={'fontsize': 8},
                cbar_kws={'label': 'ranksums statistic'},
                linewidths=0.0,
                xticklabels=1,   # force all x tick labels
                yticklabels=1,   # force all y tick labels
                ax=ax
            )

            # Explicitly set all ticks to prevent auto-skipping
            ax.set_xticks(np.arange(heat.shape[1]) + 0.5)
            ax.set_xticklabels(heat.columns, rotation=90)
            ax.set_yticks(np.arange(heat.shape[0]) + 0.5)
            ax.set_yticklabels(heat.index, rotation=0)

            ax.set_xlabel('cells')
            ax.set_ylabel(f'{module_name.title()} interaction')
            ax.set_title(f'{contrast} — statistic heatmap')
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print(f'Saved: {pdf_path}')

In [ ]:
combined_niche_ranksums.loc[combined_niche_ranksums['cells'].str.startswith('B')].sort_values('statistic', ascending=False).head(50)

In [ ]:
combined_niche_ranksums.loc[combined_niche_ranksums['cells'].str.startswith('B')].sort_values('statistic', ascending=False).tail(50)

In [ ]:
shared_niche_ranksums.loc[shared_niche_ranksums['cells'].str.startswith('B')].sort_values('statistic', ascending=False).head(50)

In [ ]:
shared_niche_ranksums.loc[shared_niche_ranksums['cells'].str.startswith('B')].sort_values('statistic', ascending=False).tail(50)